# Shareability Metric

In [10]:
import pandas as pd
from pathlib import Path
import numpy as np

In [11]:
df = pd.read_csv(Path().cwd().resolve().parent / "data/sample_data.csv")

## Collapse seperate date times into 1 date time feature

In [12]:
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

## Sort by date time to ensure t and t+1 relationship

In [13]:
df = df.sort_values("datetime", ascending=True).reset_index(drop=True)

## Drop non continous features to preserve a clean cross covariance matrix M

In [14]:
df = df.drop(columns=["year", "month", "day", "hour", "wd", "station"])

## Handle NaN values

In [15]:
df = df.dropna().reset_index(drop=True)

## Create lagged pairs

In [18]:
current = df.copy()
future = df.shift(-1)

valid_pair = (df["datetime"].shift(-1) - df["datetime"]).eq(pd.Timedelta(hours=1))

x_current = current.loc[valid_pair].reset_index(drop=True)
x_future = future.loc[valid_pair].reset_index(drop=True)

## Convert to numpy for easier tensor work

In [19]:
current_np = np.array(x_current)
future_np = np.array(x_future)

print(current_np.shape)
print(future_np.shape)

(30943, 13)
(30943, 13)


## Establish train val and test splits

In [20]:
train_pct = 0.70
val_pct = 0.10

train_current = current_np[:int(train_pct * current_np.shape[0])]
train_future = future_np[:int(train_pct * future_np.shape[0])]

val_current = current_np[int(train_pct * current_np.shape[0]):int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0])]
val_future = future_np[int(train_pct * future_np.shape[0]):int(train_pct * future_np.shape[0]) + int(val_pct * future_np.shape[0])]

test_current = current_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]
test_future = future_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]

print(train_current.shape)
print(train_future.shape)

print(val_current.shape)
print(val_future.shape)

print(test_current.shape)
print(test_future.shape)

(21660, 13)
(21660, 13)
(3094, 13)
(3094, 13)
(6189, 13)
(6189, 13)
